# Kernel customisation (notebook)

- [Using a uv virtual environment](#uv-managed-virtual-environment)
- [Storing a custom definition](#embed-the-custom-kernel-in-the-kernelspec-folder)
- [Configuration options](#configuration-options)

## uv managed virtual environment

We can use uv to create a virtual environment for different versions of Python.

Let's create a new virtual environment in the folder named ".venv_py315" using uv.

**Requires a recent version of uv to be installed.**

In [ ]:
%uv venv .venv_py315 --python 3.15 --clear
%uv pip install --directory .venv_py315 --python 3.15 async-kernel

Now we can write a kernel spec that uses uv to start the kernel from the virtual environment.

In [ ]:
import pathlib

from async_kernel.kernelspec import write_kernel_spec

uv_path = pathlib.Path.cwd().joinpath(".venv_py315")
assert uv_path.exists()

write_kernel_spec(
    name="async_3.15",
    display_name="Python 3.15 (async)",
    env={"UV_PROJECT_ENVIRONMENT": str(uv_path)},
    command=("uv", "run", "--module", "async_kernel", "start"),
)

The kernel spec with the display name "Python 3.15 (async kernel)" has been added. You will need to refresh the list of kernels for it to be available.

## Embed the custom kernel in the kernelspec folder


A callable can be passed to `write_kernel_spec` as the argument `start_interface`, which is written to the kernel spec folder and used when starting the kernel.

Let's write a kernel that will echo (print) the code written to the cell. 


In [ ]:
import async_kernel.kernelspec


def start_interface(settings: dict) -> None:

    from async_kernel import Kernel
    from async_kernel.interface.zmq import ZMQInterface

    class EchoKernel(Kernel):
        async def execute_request(self, job):
            print(job["msg"]["content"]["code"])
            return {"status": "ok", "execution_count": 0, "user_expressions": {}}

    # Subclassing the kernel will override it.

    ZMQInterface.launch_instance()


# Write the kernel spec
async_kernel.kernelspec.write_kernel_spec(name="echo", display_name="Echo kernel", start_interface=start_interface)

### Customize the shell

Sometimes it might be useful to customise the shell instead of the kernel.

Lets write the echo kernel with a bypass keyword "# bypass".

In [ ]:
import async_kernel.kernelspec


def start_interface(settings: dict) -> None:

    from traitlets import traitlets

    from async_kernel.asyncshell import AsyncInteractiveShell, AsyncInteractiveSubshell
    from async_kernel.interface.zmq import ZMQInterface

    class EchoShell(AsyncInteractiveShell):
        @traitlets.default("banner1")
        def _default_banner1(self) -> str:
            return "Echo kernel with bypass\n"

        def transform_cell(self, raw_cell):
            if raw_cell.startswith("# bypass"):
                return super().transform_cell(raw_cell)
            return f'print("""{raw_cell}""")'

    class EchoSubshell(EchoShell, AsyncInteractiveSubshell):
        pass

    ZMQInterface.launch_instance()


# Write the kernel spec
async_kernel.kernelspec.write_kernel_spec(
    name="echo-shell-with-bypass", display_name="Echo kernel with bypass", start_interface=start_interface
)

## Configuration options

Configuration is done using traitlets configuration. Configuration options are available from the command line.

In [ ]:
!async-kernel --help-all